# Chapter 5: DataFrames and Data Wrangling

**Duration: 3 hours**

## 5.1 The DataFrames.jl Package

In [ ]:
using DataFrames

# Create a DataFrame from columns
df = DataFrame(
    name   = ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    age    = [28, 35, 42, 31, 27],
    salary = [55000, 72000, 88000, 63000, 51000],
    dept   = ["Engineering", "Marketing", "Engineering", "HR", "Marketing"]
)

# Basic inspection
size(df)
nrow(df), ncol(df)
names(df)
describe(df)
first(df, 3)

## 5.2 Reading and Writing Data

In [ ]:
using CSV, DataFrames

# Read CSV files
# df = CSV.read("data.csv", DataFrame)

# Read with options
# df = CSV.read("data.csv", DataFrame;
#     delim=',',
#     header=true,
#     missingstring="NA",
#     types=Dict(:age => Int, :salary => Float64),
#     select=[:name, :age, :salary]
# )

# Write to CSV
# CSV.write("output.csv", df)

In [ ]:
# Read from RDatasets (built-in classic datasets)
using RDatasets
iris = dataset("datasets", "iris")
first(iris, 5)

## 5.3 Selecting and Filtering

In [ ]:
using DataFrames, RDatasets
iris = dataset("datasets", "iris")

# Select columns
select(iris, :SepalLength, :Species) |> first

In [ ]:
select(iris, Not(:PetalWidth)) |> first

In [ ]:
select(iris, r"Sepal") |> first

In [ ]:
# Create new columns with select
select(iris, :SepalLength, :SepalWidth,
    [:SepalLength, :SepalWidth] => ByRow((l, w) -> l * w) => :SepalArea
) |> first

In [ ]:
# Filter rows
filter(:Species => ==("setosa"), iris) |> x -> first(x, 3)

# Complex filter
filter(row -> row.SepalLength > 5.0 && row.Species == "setosa", iris) |> x -> first(x, 3)

## 5.4 Transforming Data

In [ ]:
using DataFrames, Statistics

df = DataFrame(
    city = ["Paris", "London", "Berlin", "Madrid", "Rome"],
    pop_million = [2.16, 8.98, 3.64, 3.22, 2.87],
    area_km2 = [105, 1572, 892, 604, 1285]
)

# Add a new column
transform(df, [:pop_million, :area_km2] =>
    ByRow((p, a) -> p * 1e6 / a) => :density)

In [ ]:
# Handle missing data
df_missing = DataFrame(
    x = [1, 2, missing, 4, missing],
    y = [missing, 2, 3, missing, 5]
)
dropmissing(df_missing)          # drop rows with any missing
coalesce.(df_missing.x, 0)      # replace missing with 0

## 5.5 Sorting

In [ ]:
using DataFrames, RDatasets
mtcars = dataset("datasets", "mtcars")

# Sort by one column
sort(mtcars, :MPG) |> x -> first(x, 3)

# Sort descending
sort(mtcars, :MPG, rev=true) |> x -> first(x, 3)

# Sort by multiple columns
sort(mtcars, [:Cyl, :MPG]) |> x -> first(x, 3)

## 5.6 Grouping and Split-Apply-Combine

In [ ]:
using DataFrames, Statistics, RDatasets
iris = dataset("datasets", "iris")

# Group by species
gdf = groupby(iris, :Species)

# Apply aggregation
combine(gdf,
    :SepalLength => mean => :mean_sepal_length,
    :SepalLength => std  => :std_sepal_length,
    :PetalLength => mean => :mean_petal_length,
    nrow => :count
)

In [ ]:
# Multiple grouping columns
mtcars = dataset("datasets", "mtcars")
combine(groupby(mtcars, [:Cyl, :Gear]),
    :MPG => mean => :avg_mpg,
    :HP  => mean => :avg_hp,
    nrow => :count
)

In [ ]:
# Transform within groups (keeps all rows)
transform(groupby(iris, :Species),
    :SepalLength => (x -> (x .- mean(x)) ./ std(x)) => :SepalLength_zscore
) |> x -> first(x, 5)

## 5.7 Joins

In [ ]:
using DataFrames

# Two related tables
employees = DataFrame(
    id = [1, 2, 3, 4, 5],
    name = ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    dept_id = [10, 20, 10, 30, 20]
)

departments = DataFrame(
    dept_id = [10, 20, 30, 40],
    dept_name = ["Engineering", "Marketing", "HR", "Finance"]
)

# Inner join
innerjoin(employees, departments, on=:dept_id)

In [ ]:
# Left join (keep all employees)
leftjoin(employees, departments, on=:dept_id)

In [ ]:
# Anti-join (employees NOT in sales)
sales = DataFrame(emp_id=[1,2,3], revenue=[1000, 2000, 1500])
antijoin(employees, sales, on=:id => :emp_id)

## 5.8 Reshaping: Stack and Unstack

In [ ]:
using DataFrames

# Wide format
wide = DataFrame(
    country = ["France", "Germany", "Italy"],
    pop_2020 = [67.39, 83.24, 59.55],
    pop_2021 = [67.75, 83.16, 59.24],
    pop_2022 = [67.97, 84.08, 58.86]
)

# Wide → Long (stack)
long = stack(wide, r"pop_", :country;
    variable_name=:year, value_name=:population)
println(long)

In [ ]:
# Long → Wide (unstack)
unstack(long, :country, :year, :population)

## Exercises

In [ ]:
# Exercise 1: Load iris dataset. Compute mean and std of all four
# measurements, grouped by species.
# Your code here

In [ ]:
# Exercise 2: Load mtcars. Filter HP > 100 and Cyl >= 6. Sort by MPG desc.
# Your code here

In [ ]:
# Exercise 3: Create students (id, name, major) and grades (student_id, course, grade)
# DataFrames. Inner join and compute average grade per major.
# Your code here

In [ ]:
# Exercise 4: Using mtcars, group by Cyl, compute mean MPG, mean HP, and count.
# Your code here

In [ ]:
# Exercise 5: Create wide DataFrame with monthly sales for 4 products.
# Reshape to long format, compute total annual sales per product.
# Your code here